# CPET — array-stack driver

In [ ]:
# region -> runConfig: the ONE place run configuration lives (applied before JAX init below)
runConfig = {
    # file references
    "model":       "cpet.json",   # model structure (topology)
    "scenario":    "cpet.json",   # scenario: shared/baseline/control/calibration
    "mode":        "control",     # "baseline" | "calibration" | "control"

    # device / precision (applied in Imports cell, before `import jax`)
    "device": {
        "useGpu":    False,       # CPU run (True needs jax[cuda12])
        "precision": "float64",   # "float64" or "float32"
    },

    # output / presentation
    "output":         {"save": True, 
                       "path": "data/cpet", 
                       "name": "results_CPET"
                       },
    "postProcessing": "cpet.json",   # processing config, or None to skip
    "requested":      None,          # post-proc scope: None = all signals, or a signal list
    "plots": [
        "cpet/overview.json",
        "cpet/membranes.json",
        "cpet/lung.json",
    ],
    "payload": {"signals": ["RER", "V'O2", "V'CO2", "CO"]},   # web-payload preview signals
    "printStatus": True,             # verbose run status

    # integration numerics (override scenario shared.integration)
    "runTime": 10,        # simulated seconds per internal run
    "dt":      0.00025,   # integrator step
    "dtDense": 1.0,       # save/output grid: 1.0 = 1 Hz; raise for within-beat waveforms
}
# endregion

## Imports

In [ ]:
# region -> Imports: device/precision env setup, then library + third-party imports
# ---- device / precision (from runConfig, MUST run before JAX initialises) -------
import os
if runConfig["device"]["useGpu"]:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)
    os.environ["JAX_PLATFORMS"] = "cuda"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
    os.environ["JAX_PLATFORMS"] = "cpu"

import library.run.runner as runner            # run orchestration (modes + stage stacks)
import library.run.runIO as runIO              # save/process/plot helpers (hdf5 + resultsEngine + viz)
import library.utils as utils
import numpy as np
import json
import jax

jax.config.update("jax_enable_x64", runConfig["device"]["precision"] == "float64")
np.set_printoptions(suppress=True)
print(jax.devices())
# endregion

## Assemble `simulationParams`



In [ ]:
# region -> assemble simulationParams from runConfig + loaded scenario
scenario = utils.loadScenario(runConfig["scenario"])
simulationParams = runner.buildSimulationParams(runConfig, scenario)
_integ = scenario["shared"]["integration"]
print(f"mode = {runConfig['mode']} | dt = {_integ['dt']} | dtDense = {_integ['dtDense']}"
      f" | runTime = {_integ['runTime']}")
# endregion

## Result handling


## Run

In [ ]:
# region -> Run the CPET simulation, then save raw run + config + processed signals
mode = runConfig["mode"]
print(f"Running CPET (mode={mode})")

states, modelObjects, modelStructure, results, structures = runner.run(simulationParams)

if simulationParams["saveToHDF5"]:
    runIO.saveRun(simulationParams, results, states, modelStructure)
    # /config: raw model + scenario + metadata + post-processing -> re-mountable run
    runIO.saveConfig(simulationParams, runConfig, scenario)

newResults = results
engine = None
if simulationParams["postProcessing"]["enabled"]:
    print("Processing results")
    newResults, engine = runIO.processResults(
        simulationParams, results, modelObjects, modelStructure,
        requested=runConfig.get("requested"))
    if simulationParams["saveToHDF5"]:
        # save only the pipeline-computed signals (raw leaves already live in raw/);
        # if a requested scope was set, save exactly that.
        runIO.saveProcessed(simulationParams, newResults, engine,
                      names=runConfig.get("requested"))
# endregion

In [ ]:
# region -> Plot results per configured plot files
if simulationParams["plotResults"]:
    runIO.plotResults(simulationParams, newResults, modelStructure)
# endregion

## Web payload (optional)

<details>
<summary>`engine.toPayload(requested)` serializes the chosen signals into a JSON-friendly</summary>

`{signals, units, labels, latex, errors}` dict — the shape a web backend would return to a
browser. `latex` holds the paper-quality math name for each signal (from `config/labels.json`,
via `utils.labelFor`), ready for MathJax. Uncomment the `json.dump` to write it to `notebookData/`.

</details>

In [ ]:
# Optional: JSON-friendly payload for a web client/server (the "server computes,
# returns data" path). Picks a few signals that were actually computed.
if engine is not None:
    want = [s for s in runConfig["payload"]["signals"] if s in engine.signals]
    payload = engine.toPayload(requested=want)
    print("signals:", list(payload["signals"].keys()))
    print("units:  ", payload["units"])
    print("labels: ", payload["labels"])
    print("latex:  ", payload["latex"])   # paper-quality names from config/labels.json
    print("errors: ", payload["errors"])
    # import json
    # json.dump(payload, open(os.path.join("data", "cpet_payload.json"), "w"))